# Logistic regression

In [13]:
import numpy as np
import pandas as pd

## full pipeline

In [14]:
df = pd.read_csv("train.csv")

print(df.head())
print(df.tail())

df.columns = df.columns.str.lower()
print(df.columns.tolist())
print(df.isnull().sum())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
  

In [15]:
# data cleaning
df = df.drop(columns=["passengerid", 'name', 'cabin'], errors="ignore")
df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])

df["sex"] = df["sex"].map({"female": 0, "male": 1})
df = df.copy()

print(df.isnull().sum())

survived    0
pclass      0
sex         0
age         0
sibsp       0
parch       0
ticket      0
fare        0
embarked    0
dtype: int64


## baseline model

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

base_feats = [
    'pclass',
    'sex',
    'age',
    'fare',
]

X = df[base_feats]
y = df["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

imputer = SimpleImputer(strategy="median")
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

base_model = LogisticRegression(max_iter=1000)
base_model.fit(X_train, y_train)
print("Baseline Test Accuracy:", base_model.score(X_test, y_test))

Baseline Test Accuracy: 0.8044692737430168


## feature engineering

In [17]:
# new features : family size and is alone
df["family_size"] = df['sibsp'] + df['parch'] + 1
df["is_alone"] = (df['family_size'] == 1).astype(int)



df["sex_male"] = (df["sex"] == 1).astype(int)
df["embarked_Q"] = (df["embarked"] == "Q").astype(int)
df["embarked_S"] = (df["embarked"] == "S").astype(int)
print(df.columns.tolist())

['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'embarked', 'family_size', 'is_alone', 'sex_male', 'embarked_Q', 'embarked_S']


## logistic regression model

In [18]:
feats = [
    'pclass','sex_male','age','fare', 'embarked_S','embarked_Q', 'family_size', 'is_alone'
]

X = df[feats]
y = df["survived"]

X_train , X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

imputer = SimpleImputer(strategy="median")
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
print("Test Accuracy:", model.score(X_test, y_test))

Test Accuracy: 0.7988826815642458


## predictions and confusion matrix

In [22]:
from sklearn.metrics import confusion_matrix, classification_report

# predict
y_pred = model.predict(X_test)

print("First 10 Predictions:", y_pred[:10])
print("Actual Values:  ", y_test.values[:10])

# confusion matrix evaluate
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix ", cm)

# classification report
cr = classification_report(y_test, y_pred)
print("Classification Report: \n", cr)

First 10 Predictions: [0 0 0 1 1 1 1 0 1 1]
Actual Values:   [1 0 0 1 1 1 1 0 1 1]
Confusion matrix  [[90 15]
 [21 53]]
Classification Report: 
               precision    recall  f1-score   support

           0       0.81      0.86      0.83       105
           1       0.78      0.72      0.75        74

    accuracy                           0.80       179
   macro avg       0.80      0.79      0.79       179
weighted avg       0.80      0.80      0.80       179



## making a model with different thresholds

In [23]:
# Chegara tanlash (Threshold Tuning)
from sklearn.metrics import accuracy_score

# Get predicted probabilities (chanslar) instead of hard 0 or 1
y_probs = model.predict_proba(X_test)[:, 1]

thresholds = [i / 10 for i in range(1, 10)]
results = []

for th in thresholds:
    # If probability >= threshold, predict 1 (survived), else 0
    y_pred_th = (y_probs >= th).astype(int)
    acc = accuracy_score(y_test, y_pred_th)
    results.append({"Threshold": th, "Accuracy": acc})

# Convert to DataFrame to make a nice table
df_thresholds = pd.DataFrame(results)
print(df_thresholds)
#Qaysi chegarani tanlaysiz va nega?
# odatda 0.5 olinadi shunga uni atrofdagi yaxshiro boladi, shunga 0.4 olamiz

   Threshold  Accuracy
0        0.1  0.491620
1        0.2  0.720670
2        0.3  0.804469
3        0.4  0.804469
4        0.5  0.798883
5        0.6  0.821229
6        0.7  0.804469
7        0.8  0.765363
8        0.9  0.709497


## adding 2 new features

In [24]:
from sklearn.model_selection import cross_val_score

# work with fresh copy
df_custom = df.copy()

# 1 Har bir kishiga to'g'ri keladigan pul (Fare / Family Size)
df_custom["fare_per_person"] = df_custom["fare"] / df_custom["family_size"]

# 2 Bola yoki kattaligi (16 yoshdan kichiklar - 1, qolganlar - 0)
df_custom["is_child"] = (df_custom["age"] < 16).astype(int)

#
base_feats = ['pclass','sex_male','age','fare', 'embarked_S','embarked_Q', 'family_size', 'is_alone']
X_base = df_custom[base_feats]
y = df_custom["survived"]

imputer = SimpleImputer(strategy="median")
X_base_imp = imputer.fit_transform(X_base)

cv_base = cross_val_score(LogisticRegression(max_iter=1000), X_base_imp, y, cv=5, scoring='accuracy').mean()
print(f"1. Base CV Accuracy: {cv_base:.4f}")


# 1 ta feature qoshilganda
feats_f1 = base_feats + ['fare_per_person']
X_f1 = imputer.fit_transform(df_custom[feats_f1])

cv_f1 = cross_val_score(LogisticRegression(max_iter=1000), X_f1, y, cv=5, scoring='accuracy').mean()
print(f"2. CV Accuracy (+ fare_per_person): {cv_f1:.4f}")

# 2 ta feature qoshilganda
feats_f2 = feats_f1 + ['is_child']
X_f2 = imputer.fit_transform(df_custom[feats_f2])

cv_f2 = cross_val_score(LogisticRegression(max_iter=1000), X_f2, y, cv=5, scoring='accuracy').mean()
print(f"3. CV Accuracy (+ is_child): {cv_f2:.4f}")

1. Base CV Accuracy: 0.7935
2. CV Accuracy (+ fare_per_person): 0.7935
3. CV Accuracy (+ is_child): 0.7980
